# 06 — Native Station Value Extraction (FINAL FIXED v2)

This version is linked to the corrected 00–05 workflow and the current repository folder structure.

### Main corrections
- Uses the actual precipitation folders (`CHIRPS_TIFF_2017_2022`, `IMERG_Monthly`, `ERA5_TIFF`, etc.).
- Uses **raw/native precipitation rasters** for station extraction.
- Produces one tidy row per station-year-month.
- Does **not** edge-clamp stations outside raster coverage.
- Does **not** nearest-fill NoData.
- Avoids unnecessary EPSG lookup for geographic rasters, which also prevents the earlier CDR `EngineeringCRS` issue from breaking native sampling.
- Uses the same logical feature names expected by the corrected final modelling workflow:
  `CCS, PDIR, GSMaP_MVK, CDR, CHIRPS, IMERG, GSMaP_Gauge, ERA5`.
- `CDR` is treated as PERSIANN-CDR; standalone `PERSIANN` is excluded.
- Land predictors: `DEM, NDVI, LST_Day, Distance_Sea`.


In [1]:

# ============================================================
# 06 — NATIVE STATION VALUE EXTRACTION (FINAL FIXED v2)
# Khulna Precipitation Downscaling
# ============================================================

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import rasterio
from pyproj import Transformer


# ============================================================
# 1. FIND PROJECT ROOT
# ============================================================

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "data").exists():
            return candidate

    raise FileNotFoundError(
        "Project root not found. "
        "Run this notebook from inside the repository."
    )


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("PROJECT_ROOT =", PROJECT_ROOT)


# ============================================================
# 2. SETTINGS
# ============================================================

GAUGE_PATH = (
    PROCESSED_DIR
    / "gauge_monthly_clean.csv"
)

if not GAUGE_PATH.exists():
    raise FileNotFoundError(
        "gauge_monthly_clean.csv not found. "
        "Run 02_Gauge_Preprocessing_FIXED.ipynb first."
    )


# Direct WGS84 definition to avoid EPSG database lookup
# in the current PROJ environment.
WGS84_PROJ = "+proj=longlat +datum=WGS84 +no_defs"


# IMPORTANT:
# Logical model feature names -> actual raw folder names
PRECIP_FOLDERS = {
    "CCS": "CCS",
    "PDIR": "PDIR",
    "GSMaP_MVK": "GSMaP_MVK",
    "CDR": "CDR",
    "CHIRPS": "CHIRPS_TIFF_2017_2022",
    "IMERG": "IMERG_Monthly",
    "GSMaP_Gauge": "GSMaP_Gauge_v7",
    "ERA5": "ERA5_TIFF",
}

DYNAMIC_LAND = [
    "NDVI",
    "LST_Day",
]


# ============================================================
# 3. LOAD CLEAN GAUGE DATA
# ============================================================

gauge = pd.read_csv(
    GAUGE_PATH
)

required_gauge_cols = [
    "station_id",
    "year",
    "month",
    "rainfall_mm",
    "latitude",
    "longitude",
]

missing_cols = [
    c for c in required_gauge_cols
    if c not in gauge.columns
]

if missing_cols:
    raise KeyError(
        f"Gauge file is missing required columns: {missing_cols}"
    )


print(
    "Gauge rows:",
    len(gauge)
)

print(
    "Stations:",
    gauge["station_id"].nunique()
)

print(
    "Years:",
    sorted(gauge["year"].dropna().unique().astype(int))
)


# ============================================================
# 4. HELPERS
# ============================================================

def parse_ym(name):
    """
    Parse year/month from names such as:
    product_2017_01.tif
    product-2017-01.tif
    product_201701.tif
    """

    stem = Path(name).stem

    patterns = [
        r"(?<!\d)(20\d{2})[_-](0?[1-9]|1[0-2])(?!\d)",
        r"(?<!\d)(20\d{2})(0[1-9]|1[0-2])(?!\d)",
    ]

    for pat in patterns:
        m = re.search(
            pat,
            stem
        )

        if m:
            return (
                int(m.group(1)),
                int(m.group(2))
            )

    return None


def list_rasters(folder):
    if not folder.exists():
        return []

    return sorted([
        *folder.rglob("*.tif"),
        *folder.rglob("*.tiff"),
    ])


def monthly_map(folder):
    out = {}

    for p in list_rasters(folder):

        ym = parse_ym(
            p.name
        )

        if ym is None:
            continue

        if ym in out:
            raise ValueError(
                f"Duplicate raster for {folder.name} {ym}:\n"
                f"{out[ym]}\n{p}"
            )

        out[ym] = p

    return out


def choose_static(
    folder_name,
    preferred_names
):

    folder = (
        RAW_DIR
        / "predictors"
        / folder_name
    )

    files = list_rasters(
        folder
    )

    if not files:
        raise FileNotFoundError(
            f"No raster found for {folder_name}"
        )

    lookup = {
        p.name.lower(): p
        for p in files
    }

    # Prefer exact known filenames
    for name in preferred_names:
        if name.lower() in lookup:
            return lookup[
                name.lower()
            ]

    # Avoid obvious intermediate products
    clean = [
        p for p in files
        if not any(
            k in p.stem.lower()
            for k in [
                "clip",
                "tmp",
                "temp",
                "aligned",
                "resampl",
            ]
        )
    ]

    if len(clean) == 1:
        return clean[0]

    if len(files) == 1:
        return files[0]

    raise ValueError(
        f"Ambiguous static predictor {folder_name}:\n"
        + "\n".join(
            str(p)
            for p in files
        )
    )


def bounds_look_geographic(bounds):
    return (
        -180 <= bounds.left <= 180
        and -180 <= bounds.right <= 180
        and -90 <= bounds.bottom <= 90
        and -90 <= bounds.top <= 90
    )


def sample_native(
    path,
    lon,
    lat
):
    """
    Sample one raw/native raster at a gauge coordinate.

    Important safeguards:
    - No edge clamping.
    - No nearest filling.
    - NoData -> NaN.
    - If raster bounds are clearly lon/lat, station coordinates
      are sampled directly, avoiding unnecessary PROJ operations.
    - For genuinely projected rasters, coordinate transformation
      is attempted using the raster's own CRS.
    """

    with rasterio.open(
        path
    ) as src:

        b = src.bounds

        # ----------------------------------------------------
        # CASE A:
        # Raster grid itself clearly uses lon/lat coordinates.
        # This also safely handles CDR EngineeringCRS metadata.
        # ----------------------------------------------------

        if bounds_look_geographic(
            b
        ):

            x = float(
                lon
            )

            y = float(
                lat
            )


        # ----------------------------------------------------
        # CASE B:
        # Raster is genuinely projected.
        # ----------------------------------------------------

        else:

            if src.crs is None:
                return (
                    np.nan,
                    "crs_missing_non_geographic"
                )

            try:

                transformer = Transformer.from_crs(
                    WGS84_PROJ,
                    src.crs,
                    always_xy=True
                )

                x, y = transformer.transform(
                    float(lon),
                    float(lat)
                )

            except Exception as e:

                warnings.warn(
                    f"Coordinate transformation failed for "
                    f"{path.name}: {e}"
                )

                return (
                    np.nan,
                    "transform_failed"
                )


        # ----------------------------------------------------
        # No edge clamping:
        # outside coverage stays missing.
        # ----------------------------------------------------

        if not (
            b.left <= x <= b.right
            and b.bottom <= y <= b.top
        ):

            return (
                np.nan,
                "outside"
            )


        # ----------------------------------------------------
        # Convert coordinate to raster row/column
        # ----------------------------------------------------

        try:
            row, col = src.index(
                x,
                y
            )

        except Exception:

            return (
                np.nan,
                "index_failed"
            )


        if (
            row < 0
            or row >= src.height
            or col < 0
            or col >= src.width
        ):

            return (
                np.nan,
                "outside"
            )


        # ----------------------------------------------------
        # Read exactly the native source pixel
        # ----------------------------------------------------

        a = src.read(
            1,
            window=(
                (
                    row,
                    row + 1
                ),
                (
                    col,
                    col + 1
                )
            ),
            masked=True
        )


        if a.size == 0:
            return (
                np.nan,
                "empty"
            )


        if np.ma.is_masked(
            a[0, 0]
        ):

            return (
                np.nan,
                "nodata"
            )


        val = float(
            a[0, 0]
        )


        if not np.isfinite(
            val
        ):

            return (
                np.nan,
                "nodata"
            )


        if (
            src.nodata is not None
            and np.isclose(
                val,
                src.nodata
            )
        ):

            return (
                np.nan,
                "nodata"
            )


        return (
            val,
            "ok"
        )


# ============================================================
# 5. PARSER SELF-TEST
# ============================================================

parser_tests = {
    "CCS_2022_01.tif": (2022, 1),
    "abc-2021-12.tif": (2021, 12),
    "IMERG_202203.tif": (2022, 3),
}

for test_name, expected in parser_tests.items():

    got = parse_ym(
        test_name
    )

    if got != expected:
        raise RuntimeError(
            f"Year/month parser failed for {test_name}. "
            f"Got {got}, expected {expected}"
        )


print(
    "Filename year-month parser: OK"
)


# ============================================================
# 6. CANONICAL STATIC PREDICTORS
# ============================================================

DEM_PATH = choose_static(
    "DEM",
    [
        "Khulna_SRTM_DEM.tif",
        "DEM.tif",
    ],
)


DFS_PATH = choose_static(
    "Distance_Sea",
    [
        "Distance_Sea.tif",
        "distance_to_sea.tif",
    ],
)


print(
    "\nCanonical DEM:"
)

print(
    DEM_PATH
)


print(
    "\nCanonical Distance to Sea:"
)

print(
    DFS_PATH
)


# ============================================================
# 7. BUILD MONTHLY SOURCE MAPS
# ============================================================

precip_maps = {}

print(
    "\n=========================================="
)

print(
    "PRECIPITATION SOURCE CHECK"
)

print(
    "=========================================="
)


for feature_name, folder_name in (
    PRECIP_FOLDERS.items()
):

    folder = (
        RAW_DIR
        / "precipitation"
        / folder_name
    )


    if not folder.exists():

        raise FileNotFoundError(
            f"Missing precipitation folder:\n{folder}"
        )


    mm = monthly_map(
        folder
    )


    precip_maps[
        feature_name
    ] = mm


    print(
        f"{feature_name:14s} | "
        f"folder={folder_name:24s} | "
        f"parsed={len(mm)}"
    )


    missing = [
        (
            y,
            m
        )
        for y in range(
            2017,
            2023
        )
        for m in range(
            1,
            13
        )
        if (
            y,
            m
        ) not in mm
    ]


    if missing:

        raise FileNotFoundError(
            f"{feature_name} is missing months:\n"
            f"{missing}"
        )


# ============================================================
# 8. DYNAMIC LAND MAPS
# ============================================================

land_maps = {}


print(
    "\n=========================================="
)

print(
    "DYNAMIC LAND SOURCE CHECK"
)

print(
    "=========================================="
)


for feature_name in DYNAMIC_LAND:

    folder = (
        RAW_DIR
        / "predictors"
        / feature_name
    )


    if not folder.exists():

        raise FileNotFoundError(
            f"Missing predictor folder:\n{folder}"
        )


    mm = monthly_map(
        folder
    )


    land_maps[
        feature_name
    ] = mm


    print(
        f"{feature_name:14s} | "
        f"parsed={len(mm)}"
    )


    missing = [
        (
            y,
            m
        )
        for y in range(
            2017,
            2023
        )
        for m in range(
            1,
            13
        )
        if (
            y,
            m
        ) not in mm
    ]


    if missing:

        raise FileNotFoundError(
            f"{feature_name} is missing months:\n"
            f"{missing}"
        )


# ============================================================
# 9. SAMPLE STATIC VARIABLES ONCE PER STATION
# ============================================================

station_static = {}
static_status = []


for station, s in gauge.groupby(
    "station_id",
    sort=False
):

    lon = float(
        s["longitude"].median()
    )

    lat = float(
        s["latitude"].median()
    )


    dem, st_dem = sample_native(
        DEM_PATH,
        lon,
        lat
    )


    dfs, st_dfs = sample_native(
        DFS_PATH,
        lon,
        lat
    )


    station_static[
        station
    ] = {
        "DEM": dem,
        "Distance_Sea": dfs,
    }


    static_status.append(
        {
            "station_id": station,
            "DEM_status": st_dem,
            "Distance_Sea_status": st_dfs,
        }
    )


static_status_df = pd.DataFrame(
    static_status
)


print(
    "\nStatic predictor extraction status:"
)

display(
    static_status_df
)


# ============================================================
# 10. BUILD TIDY NATIVE STATION-MONTH TABLE
# ============================================================

records = []
qc_records = []


for _, r in gauge.iterrows():

    station = r[
        "station_id"
    ]

    y = int(
        r[
            "year"
        ]
    )

    m = int(
        r[
            "month"
        ]
    )

    lon = float(
        r[
            "longitude"
        ]
    )

    lat = float(
        r[
            "latitude"
        ]
    )


    rec = {
        "station_id": station,
        "year": y,
        "month": m,
        "date": f"{y}-{m:02d}-01",
        "latitude": lat,
        "longitude": lon,
        "rainfall_mm": float(
            r[
                "rainfall_mm"
            ]
        ),
    }


    rec.update(
        station_static[
            station
        ]
    )


    # --------------------------------------------------------
    # Native precipitation extraction
    # --------------------------------------------------------

    for feature_name in (
        PRECIP_FOLDERS.keys()
    ):

        path = precip_maps[
            feature_name
        ].get(
            (
                y,
                m
            )
        )


        if path is None:

            val = np.nan

            status = (
                "missing_file"
            )


        else:

            val, status = sample_native(
                path,
                lon,
                lat
            )


        # Precipitation must not be negative
        if (
            np.isfinite(
                val
            )
            and val < 0
        ):

            val = np.nan

            status = (
                "negative_invalid"
            )


        rec[
            feature_name
        ] = val


        qc_records.append(
            {
                "station_id": station,
                "year": y,
                "month": m,
                "feature": feature_name,
                "status": status,
                "source": str(path) if path is not None else None,
            }
        )


    # --------------------------------------------------------
    # Dynamic land extraction
    # --------------------------------------------------------

    for feature_name in (
        DYNAMIC_LAND
    ):

        path = land_maps[
            feature_name
        ].get(
            (
                y,
                m
            )
        )


        if path is None:

            val = np.nan

            status = (
                "missing_file"
            )


        else:

            val, status = sample_native(
                path,
                lon,
                lat
            )


        rec[
            feature_name
        ] = val


        qc_records.append(
            {
                "station_id": station,
                "year": y,
                "month": m,
                "feature": feature_name,
                "status": status,
                "source": str(path) if path is not None else None,
            }
        )


    records.append(
        rec
    )


# ============================================================
# 11. CREATE OUTPUT TABLES
# ============================================================

samples = pd.DataFrame(
    records
).sort_values(
    [
        "station_id",
        "year",
        "month",
    ]
).reset_index(
    drop=True
)


qc = pd.DataFrame(
    qc_records
)


print(
    "\n=========================================="
)

print(
    "NATIVE STATION EXTRACTION COMPLETE"
)

print(
    "=========================================="
)


print(
    "Tidy modelling table:",
    samples.shape
)


display(
    samples.head()
)


# ============================================================
# 12. EXPECTED TABLE SIZE CHECK
# ============================================================

expected_rows = len(
    gauge
)


if len(
    samples
) != expected_rows:

    raise RuntimeError(
        f"Expected {expected_rows} station-month rows "
        f"but created {len(samples)}."
    )


print(
    "Expected station-month rows:",
    expected_rows
)

print(
    "Created station-month rows:",
    len(samples)
)


# ============================================================
# 13. MISSING-VALUE QC
# ============================================================

required_features = (
    list(
        PRECIP_FOLDERS.keys()
    )
    + [
        "DEM",
        "NDVI",
        "LST_Day",
        "Distance_Sea",
    ]
)


missing_count = (
    samples[
        required_features
    ]
    .isna()
    .sum()
)


missing_pct = (
    samples[
        required_features
    ]
    .isna()
    .mean()
    * 100.0
)


missing_qc = pd.DataFrame(
    {
        "missing_count": missing_count,
        "missing_pct": missing_pct,
    }
)


print(
    "\nMissing values by feature:"
)

display(
    missing_qc
)


# ============================================================
# 14. EXTRACTION STATUS QC
# ============================================================

status_summary = (
    qc.groupby(
        [
            "feature",
            "status",
        ]
    )
    .size()
    .rename(
        "n"
    )
    .reset_index()
)


print(
    "\nExtraction status:"
)

display(
    status_summary
)


# ============================================================
# 15. IMPORTANT COVERAGE WARNINGS
# ============================================================

outside_or_failed = qc[
    qc[
        "status"
    ].isin(
        [
            "outside",
            "nodata",
            "transform_failed",
            "crs_missing_non_geographic",
            "index_failed",
            "empty",
            "negative_invalid",
        ]
    )
]


if len(
    outside_or_failed
) > 0:

    print(
        "\nWARNING:"
        " Some station-month predictor values could not be "
        "sampled from native source coverage."
    )

    print(
        "These values remain NaN. "
        "They are NOT edge-clamped or nearest-filled."
    )

    display(
        outside_or_failed.head(
            100
        )
    )


high_missing = (
    missing_qc[
        missing_qc[
            "missing_pct"
        ] > 20
    ]
)


if len(
    high_missing
) > 0:

    print(
        "\nWARNING:"
        " One or more required features have >20% missing values."
    )

    print(
        "Review source coverage/CRS before final modelling. "
        "Do not fill large gaps with nearest values."
    )

    display(
        high_missing
    )


# ============================================================
# 16. SAVE OUTPUTS
# ============================================================

samples_path = (
    PROCESSED_DIR
    / "station_samples_native_tidy.csv"
)


qc_path = (
    PROCESSED_DIR
    / "station_extraction_qc.csv"
)


missing_qc_path = (
    PROCESSED_DIR
    / "station_extraction_missing_qc.csv"
)


static_qc_path = (
    PROCESSED_DIR
    / "station_static_extraction_qc.csv"
)


samples.to_csv(
    samples_path,
    index=False
)


qc.to_csv(
    qc_path,
    index=False
)


missing_qc.to_csv(
    missing_qc_path
)


static_status_df.to_csv(
    static_qc_path,
    index=False
)


# ============================================================
# 17. FINAL SUMMARY
# ============================================================

print(
    "\n=========================================="
)

print(
    "06 NATIVE STATION VALUE EXTRACTION DONE"
)

print(
    "=========================================="
)


print(
    "Saved modelling table:",
    samples_path
)


print(
    "Saved extraction QC:",
    qc_path
)


print(
    "Saved missing-value QC:",
    missing_qc_path
)


print(
    "Saved static QC:",
    static_qc_path
)


print(
    "\nIMPORTANT:"
)

print(
    "Training/validation/test station values were sampled "
    "from RAW/NATIVE rasters, not from the 2022 aligned-grid outputs."
)

print(
    "Standalone PERSIANN is not used. "
    "CDR is the PERSIANN-CDR feature."
)

print(
    "Logical GSMaP feature name is GSMaP_Gauge; "
    "its actual raw folder is GSMaP_Gauge_v7."
)


PROJECT_ROOT = E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh
Gauge rows: 432
Stations: 6
Years: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
Filename year-month parser: OK

Canonical DEM:
E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\raw\predictors\DEM\Khulna_SRTM_DEM.tif

Canonical Distance to Sea:
E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\raw\predictors\Distance_Sea\Distance_Sea.tif

PRECIPITATION SOURCE CHECK
CCS            | folder=CCS                      | parsed=72
PDIR           | folder=PDIR                     | parsed=72
GSMaP_MVK      | folder=GSMaP_MVK                | parsed=72
CDR            | folder=CDR                      | parsed=72
CHIRPS         | folder=CHIR

,station_id,DEM_status,Distance_Sea_status
0,CL503,ok,ok
1,CL504,ok,ok
2,CL509,ok,nodata
3,CL510,ok,ok
4,CL515,ok,ok
5,CL517,ok,ok



NATIVE STATION EXTRACTION COMPLETE
Tidy modelling table: (432, 19)


,station_id,year,month,date,latitude,longitude,rainfall_mm,DEM,Distance_Sea,CCS,PDIR,GSMaP_MVK,CDR,CHIRPS,IMERG,GSMaP_Gauge,ERA5,NDVI,LST_Day
0,CL503,2017,1,2017-01-01,22.6012,89.5195,0.0,5.0,32.322636,0.0,1.0,1.966016,NaN,4.258131,0.000,0.000000,0.208773,0.3041,23.28
1,CL503,2017,2,2017-02-01,22.6012,89.5195,0.0,5.0,32.322636,0.0,0.0,0.000000,NaN,5.859640,0.003,0.193933,3.796070,0.3495,27.74
2,CL503,2017,3,2017-03-01,22.6012,89.5195,345.0,5.0,32.322636,2.0,17.0,49.963665,NaN,47.325570,0.097,117.178435,101.051186,0.3115,29.21
3,CL503,2017,4,2017-04-01,22.6012,89.5195,270.0,5.0,32.322636,31.0,56.0,334.475193,NaN,93.311977,0.155,132.086874,85.262344,0.3005,31.95
4,CL503,2017,5,2017-05-01,22.6012,89.5195,783.0,5.0,32.322636,60.0,122.0,134.737306,NaN,149.536093,0.166,152.076353,107.824173,0.4503,32.39


Expected station-month rows: 432
Created station-month rows: 432

Missing values by feature:


...


Extraction status:


,feature,status,n
0,CCS,nodata,72
1,CCS,ok,360
2,CDR,nodata,72
3,CDR,ok,360
4,CHIRPS,ok,432
5,ERA5,ok,432
6,GSMaP_Gauge,ok,432
7,GSMaP_MVK,ok,414
8,GSMaP_MVK,outside,18
9,IMERG,ok,432



These values remain NaN. They are NOT edge-clamped or nearest-filled.


,station_id,year,month,feature,status,source
3,CL503,2017,1,CDR,nodata,E:\Geospatial\Precipitation-Downscaling-Khulna...
13,CL503,2017,2,CDR,nodata,E:\Geospatial\Precipitation-Downscaling-Khulna...
23,CL503,2017,3,CDR,nodata,E:\Geospatial\Precipitation-Downscaling-Khulna...
33,CL503,2017,4,CDR,nodata,E:\Geospatial\Precipitation-Downscaling-Khulna...
43,CL503,2017,5,CDR,nodata,E:\Geospatial\Precipitation-Downscaling-Khulna...
...,...,...,...,...,...,...
1169,CL504,2020,9,LST_Day,nodata,E:\Geospatial\Precipitation-Downscaling-Khulna...
1269,CL504,2021,7,LST_Day,nodata,E:\Geospatial\Precipitation-Downscaling-Khulna...
1279,CL504,2021,8,LST_Day,nodata,E:\Geospatial\Precipitation-Downscaling-Khulna...
1379,CL504,2022,6,LST_Day,nodata,E:\Geospatial\Precipitation-Downscaling-Khulna...



06 NATIVE STATION VALUE EXTRACTION DONE
Saved modelling table: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\processed\station_samples_native_tidy.csv
Saved extraction QC: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\processed\station_extraction_qc.csv
Saved missing-value QC: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\processed\station_extraction_missing_qc.csv
Saved static QC: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\processed\station_static_extraction_qc.csv

IMPORTANT:
Training/validation/test station values were sampled from RAW/NATIVE rasters, not from the 2022 aligned-grid outputs.
Standalone PERSIANN is not used. CDR is the PER